In [54]:
from src.post_processing import PathWrangler
import polars as pl
from pathlib import Path
from collections import defaultdict

In [55]:
study = "/home/stef/quest_data/bottle/data/processed/lin_test"
known = "/home/stef/bottle/artifacts/known"
out_dir = "/home/stef/bottle/artifacts/coa_mutase_paths"

pw = PathWrangler(Path(study), Path(known))

In [56]:
tables = pw.get_paths()

In [75]:
paths = tables['paths']
krs = tables['known_reactions']
prs = tables['predicted_reactions']

pr2krs = dict(zip(prs['id'], prs['analogue_ids'].to_list()))
krs2enzymes = dict(zip(krs['id'], krs['enzymes'].to_list()))
prs2enzymes = defaultdict(list)
for p, ks in pr2krs.items():
    for k in ks:
        prs2enzymes[p] += krs2enzymes[k]

prs2smarts = dict(zip(prs['id'], prs['smarts'].to_list()))

paths = paths.select(
    pl.col("id"),
    pl.col("rxn_id"),
    (pl.col("generation") + 1).alias("step"),
    pl.col("starters").map_elements(lambda x : ";".join(x), return_dtype=pl.String),
    pl.col("targets").map_elements(lambda x : ";".join(x), return_dtype=pl.String),
    pl.col("rxn_id").replace_strict(prs2enzymes, default=[]).map_elements(lambda x : ";".join(x), return_dtype=pl.String).alias("enzymes"),
    pl.col("rxn_id").replace_strict(prs2smarts, default=None).alias("smarts")
)
paths

id,rxn_id,step,starters,targets,enzymes,smarts
str,str,i32,str,str,str,str
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""5582845919356f447668dd0ea4647f…",1,"""pyruvate""","""3hpa""","""A9WGE2;A9WC35;Q8N0X4;S5N020;Q8…","""CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C…"
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""8c16137b4045f00069ba91d38a3b8b…",2,"""pyruvate""","""3hpa""","""A9WC41;Q9JLZ3;Q13825;Q1ZXF1;F1…","""CC(O)(CC(=O)SCCNC(=O)CCNC(=O)C…"
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""9c2c777c0fea421a26efc26857522f…",3,"""pyruvate""","""3hpa""","""Q8VCH6;Q15392;Q5BQE6;Q60HC5;Q3…","""CC(=CC(=O)SCCNC(=O)CCNC(=O)C(O…"
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""e86048734a2f69f1257bd92b62ddb4…",4,"""pyruvate""","""3hpa""","""Q5KUG0;A7IQE6;Q1LRY0;A7IQE5""","""CC(CC(=O)SCCNC(=O)CCNC(=O)C(O)…"
"""0a24a1c33085a4f1ea7f8e19ce06c3…","""bd616dde067e2786bec3a4ea2c6e17…",5,"""pyruvate""","""3hpa""","""Q0P5J1;Q96K12;Q7TNT2;Q0P5J1;Q9…","""NC(=O)C1=CN(C2OC(COP(=O)(O)OP(…"
"""9b13db2d9c8b6a2b40034ef19b269b…","""27450e93bf1eb2d982a541e9b34724…",1,"""glutamic_acid""","""3hpa""","""Q24SG9;Q8RHY7;P80078;Q73KI2;Q5…","""NC(CCC(=O)O)C(=O)O>>CC(C(=O)O)…"
"""9b13db2d9c8b6a2b40034ef19b269b…","""646032df080e79e8522e14d05d8548…",2,"""glutamic_acid""","""3hpa""","""A9X6P9;P76518;B0MC58;G2SYC0;O0…","""CC(C(=O)O)C(N)C(=O)O.CC(=O)SCC…"
"""9b13db2d9c8b6a2b40034ef19b269b…","""62277981d883290bf5c0d4ed258555…",3,"""glutamic_acid""","""3hpa""","""""","""NC(=O)C1=CN(C2OC(COP(=O)(O)OP(…"
"""9b13db2d9c8b6a2b40034ef19b269b…","""e86048734a2f69f1257bd92b62ddb4…",4,"""glutamic_acid""","""3hpa""","""Q5KUG0;A7IQE6;Q1LRY0;A7IQE5""","""CC(CC(=O)SCCNC(=O)CCNC(=O)C(O)…"


In [62]:
enzymes = tables['enzymes']


enzymes.with_columns(
    pl.col("id").map_elements(lambda x : f"https://www.uniprot.org/uniprotkb/{x}/entry", return_dtype=pl.String).alias("link"),
)

id,sequence,existence,reviewed,ec,organism,name,subunit,link
str,str,enum,enum,str,str,str,bool,str
"""A0A0A1C3I2""","""MDVRRRLPPKLRRPLPITESSHHHRKTPFP…","""Evidence at transcript level""","""reviewed""","""1.1.1.34""","""Panax ginseng (Korean ginseng)""","""3-hydroxy-3-methylglutaryl coe…",false,"""https://www.uniprot.org/unipro…"
"""A0A0A1C930""","""MDVRRRPVKSLSSAKTATAGEPPKSQQQHP…","""Evidence at transcript level""","""reviewed""","""1.1.1.34""","""Panax ginseng (Korean ginseng)""","""3-hydroxy-3-methylglutaryl coe…",false,"""https://www.uniprot.org/unipro…"
"""A7Z064""","""MLSRLFRMHGLFVASHPWEVIVGTVTLTIC…","""Evidence at transcript level""","""reviewed""","""1.1.1.34""","""Bos taurus (Bovine)""","""3-hydroxy-3-methylglutaryl-coe…",false,"""https://www.uniprot.org/unipro…"
"""A9WGE2""","""MEAVTIVDVAPRDGLQNEPDVLEPATRVEL…","""Evidence at protein level""","""reviewed""","""4.1.3.46""","""Chloroflexus aurantiacus (stra…","""(R)-citramalyl-CoA lyase (EC 4…",false,"""https://www.uniprot.org/unipro…"
"""B2GV06""","""MAALKLLSSGLRLCASARNSRGALHKGCAC…","""Evidence at protein level""","""reviewed""","""2.8.3.5""","""Rattus norvegicus (Rat)""","""Succinyl-CoA:3-ketoacid coenzy…",false,"""https://www.uniprot.org/unipro…"
…,…,…,…,…,…,…,…,…
"""Q5V468""","""MGALDDLRVLDLTQVLAGPYCTMLLADMGA…","""Evidence at protein level""","""reviewed""","""2.8.3.26""","""Haloarcula marismortui (strain…","""Succinyl-CoA:mesaconate CoA-tr…",false,"""https://www.uniprot.org/unipro…"
"""Q9V1R3""","""MNVEDIIEKVANGEIKLHQVEKYVNGDKRL…","""Inferred from homology""","""reviewed""","""1.1.1.34""","""Pyrococcus abyssi (strain GE5 …","""3-hydroxy-3-methylglutaryl-coe…",false,"""https://www.uniprot.org/unipro…"
"""Q9YAS4""","""MGSSSGQKPRRLEDLVDKLASGSLSHSRLE…","""Inferred from homology""","""reviewed""","""1.1.1.34""","""Aeropyrum pernix (strain ATCC …","""3-hydroxy-3-methylglutaryl-coe…",false,"""https://www.uniprot.org/unipro…"


In [76]:
paths.write_csv(
    Path(out_dir) / "250912_3hpa_paths.csv",
    separator=','
)

enzymes.write_csv(
    Path(out_dir) / "250912_3hpa_enzymes.csv",
    separator=','
)